# 31. Hybrid Feature Fusion + Cross-Encoder

29번의 통합 데이터, HS6 적격 기준, 제품군 정의, Leave-One-Product-Family-Out 평가를 그대로 재사용합니다.

이번 실험은 SBERT 의미 검색에 제형·포장·용도 기반 구조화 특징을 결합하고, 다국어 Cross-Encoder로 후보 사례를 재평가합니다.

- 평가: 기존 LOPO, 동일 평가 모집단 및 HS6 클래스
- 구조화 특징: form_focus_text + discriminative_text
- 후보 검색: SBERT Product/Ruling + 구조화 TF-IDF
- 재순위화: mMARCO multilingual Cross-Encoder
- 비교: SBERT 기준선, Hybrid RRF, Cross-Encoder, Hybrid Cross-Encoder
- 최우선 지표: HS6 Macro-F1

평가 Fold는 후보 검색과 재순위화에서 제외하며, 파라미터는 고정해 평가 데이터에 대한 튜닝을 방지합니다.

In [ ]:
!pip -q install -U sentence-transformers kiwipiepy scipy

import json, requests, re, zipfile
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from sentence_transformers import CrossEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import LeaveOneGroupOut

SOURCE_URL = "https://raw.githubusercontent.com/lin1118-code/HS-complete-product/feature/explanatory-note-integration/29_discriminative_features_experiment.ipynb"
source_nb = requests.get(SOURCE_URL).json()
for i in [3,5,7,9,11]:
    exec("".join(source_nb["cells"][i]["source"]), globals())

print("기준 데이터:", len(eval_df), "| HS6:", len(labels), "| 제품군:", eval_df["product_family"].nunique())

CROSS_ENCODER_NAME = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, max_length=384)
logo = LeaveOneGroupOut()
splits = list(logo.split(eval_df, eval_df["hs6"], groups=eval_df["product_family"]))
prediction_rows, fold_rows = [], []

def normalized_rank(rank):
    values = pd.Series(rank, dtype=float)
    if len(values) == 0: return values
    lo, hi = values.min(), values.max()
    return (values - lo) / (hi - lo) if hi > lo else pd.Series(1.0, index=values.index)

def code_score_from_cases(scores, codes, method="max"):
    tmp = pd.DataFrame({"hs6": np.asarray(codes).astype(str), "score": np.asarray(scores, dtype=float)})
    grouped = tmp.groupby("hs6")["score"]
    result = grouped.max() if method == "max" else grouped.apply(lambda x: x.nlargest(min(3, len(x))).mean())
    return result.reindex(labels, fill_value=-1).sort_values(ascending=False)

def top_codes(rank, n=3):
    return [str(x) for x in rank.index[:n]]

def evaluate(part):
    yt, yp = part["true_hs6"].astype(str), part["predicted_hs6"].astype(str)
    top3 = np.mean([str(t) in [str(x) for x in c] for t, c in zip(yt, part["top3_candidates"])])
    return {
        "n": len(part),
        "hs4_accuracy": np.mean(yt.str[:4].to_numpy() == yp.str[:4].to_numpy()),
        "hs6_accuracy": accuracy_score(yt, yp),
        "macro_f1_hs6": f1_score(yt, yp, labels=labels, average="macro", zero_division=0),
        "weighted_f1_hs6": f1_score(yt, yp, average="weighted", zero_division=0),
        "top3_hs6_accuracy": top3
    }

for fold, (tr, te) in enumerate(splits, 1):
    train, test = eval_df.iloc[tr].copy(), eval_df.iloc[te].copy()
    train_idx = np.asarray(tr)
    train_codes = train["hs6"].astype(str).to_numpy()
    family = test["product_family"].iloc[0]
    if not set(test["hs6"]).issubset(set(train_codes)):
        raise ValueError(f"Fold {fold}: 학습 데이터에 없는 HS6 코드")
    fold_rows.append({"fold": fold, "heldout_family": family, "train_n": len(train), "test_n": len(test)})

    struct_train = (train["form_focus_text"].astype(str) + " " + train["discriminative_text"].astype(str)).tolist()
    struct_test = (test["form_focus_text"].astype(str) + " " + test["discriminative_text"].astype(str)).tolist()
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(2,5), min_df=1, max_features=30000, sublinear_tf=True)
    x_train = vectorizer.fit_transform(struct_train)
    x_test = vectorizer.transform(struct_test)

    for global_i, struct_vec in zip(te, x_test):
        q = query_emb[global_i]
        product_rank = rank_case_codes(product_emb[train_idx] @ q, train_codes)
        ruling_rank = rank_case_codes(ruling_emb[train_idx] @ q, train_codes)
        sbert_rank = weighted_rrf_rank(product_rank, ruling_rank, k=60, kb_weight=1.0)
        struct_scores = cosine_similarity(struct_vec, x_train).ravel()
        struct_rank = rank_case_codes(struct_scores, train_codes)
        hybrid_rank = weighted_rrf_rank(sbert_rank, struct_rank, k=60, kb_weight=0.35)

        candidate_ids = set()
        for rank in [sbert_rank, struct_rank, hybrid_rank]:
            candidate_ids.update(np.argsort(-np.asarray([rank.get(c, -1) for c in train_codes]))[:25].tolist())
        candidate_ids = sorted(candidate_ids, key=lambda j: hybrid_rank.get(train_codes[j], -1), reverse=True)[:40]

        pairs = [(str(eval_df.iloc[global_i]["query_text"]), str(train.iloc[j]["ruling_text"])) for j in candidate_ids]
        ce_scores = cross_encoder.predict(pairs, batch_size=32, show_progress_bar=False)
        ce_case_rank = code_score_from_cases(ce_scores, train_codes[candidate_ids], method="top3")
        ce_hybrid_rank = code_score_from_cases(
            np.asarray([0.65 * normalized_rank(pd.Series(ce_scores, index=candidate_ids)).get(j, 0) +
                        0.35 * normalized_rank(pd.Series([hybrid_rank.get(train_codes[j], 0) for j in candidate_ids], index=candidate_ids)).get(j, 0)
                        for j in candidate_ids]),
            train_codes[candidate_ids], method="top3"
        )

        for model_name, rank in [
            ("SBERT_product_plus_ruling", sbert_rank),
            ("Hybrid_Structure_RRF", hybrid_rank),
            ("CrossEncoder_Reranking", ce_case_rank),
            ("Hybrid_CrossEncoder", ce_hybrid_rank)
        ]:
            prediction_rows.append({
                "row_index": int(global_i), "fold": fold, "heldout_family": family,
                "data_source": eval_df.iloc[global_i]["data_source"],
                "true_hs6": str(eval_df.iloc[global_i]["hs6"]),
                "true_hs4": str(eval_df.iloc[global_i]["hs4"]),
                "model": model_name,
                "predicted_hs6": str(rank.index[0]),
                "top3_candidates": top_codes(rank)
            })

predictions = pd.DataFrame(prediction_rows)
fold_audit = pd.DataFrame(fold_rows)
summary = pd.DataFrame([{"model": name, **evaluate(part)} for name, part in predictions.groupby("model")]).sort_values("macro_f1_hs6", ascending=False)
class_rows = []
for model_name, part in predictions.groupby("model"):
    yt, yp = part["true_hs6"].astype(str), part["predicted_hs6"].astype(str)
    p, r, f, s = precision_recall_fscore_support(yt, yp, labels=labels, zero_division=0)
    for code, pv, rv, fv, sv in zip(labels, p, r, f, s):
        class_rows.append({"model": model_name, "hs6": code, "support": int(sv), "precision": pv, "recall": rv, "f1": fv})
class_metrics = pd.DataFrame(class_rows)
family_metrics = predictions.groupby(["model", "heldout_family"]).apply(lambda x: pd.Series(evaluate(x)), include_groups=False).reset_index()
display(summary.round(4))
display(class_metrics.sort_values(["model", "f1"]).round(4))
display(family_metrics.round(4))

OUT = Path("/content/31_hybrid_crossencoder_results")
OUT.mkdir(parents=True, exist_ok=True)
tables = {
    "31_oof_predictions.csv": predictions,
    "31_model_comparison.csv": summary,
    "31_class_metrics.csv": class_metrics,
    "31_family_metrics.csv": family_metrics,
    "31_fold_audit.csv": fold_audit
}
for name, data in tables.items():
    data.to_csv(OUT / name, index=False, encoding="utf-8-sig")
with zipfile.ZipFile("/content/31_hybrid_crossencoder_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in OUT.glob("*.csv"):
        z.write(f, arcname=f.name)
print("완료: /content/31_hybrid_crossencoder_results.zip")
